# CityPulse AI: Incident Priority Neural Network Classifier

This notebook trains a Deep Learning Multi-Layer Perceptron (MLP) using **TensorFlow & Keras** to classify municipal incident priorities (`critical`, `high`, `medium`, `low`) based on features extracted from reports: 
1. **Incident Type** (One-Hot Encoded)
2. **Ward Name** (One-Hot Encoded)
3. **Incident Description** (Processed using TF-IDF Text Vectorization)

## Colab GPU Setup Instructions
To use GPU acceleration for training:
1. In the top menu, select **Runtime** -> **Change runtime type**.
2. Under **Hardware accelerator**, select **GPU** (e.g., T4 GPU).
3. Click **Save**.
4. Execute the cells below.

### 1. Verify GPU Availability in TensorFlow

In [ ]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print("\n✅ GPU Acceleration is ENABLED!")
    for gpu in gpus:
        print(f"  - Found GPU device: {gpu}")
else:
    print("\nℹ️ GPU is not active. Using CPU. (To enable GPU, go to Runtime -> Change runtime type)")

### 2. Import Libraries

In [ ]:
import os
import random
import pickle
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

### 3. Generate Training Dataset
We generate a synthetic dataset (~1,200 samples) mapped to municipal template definitions for the neural network to learn priorities.

In [ ]:
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

types_and_wards = {
    'fire': {
        'priority': 'critical',
        'templates': [
            "Severe fire outbreak in a residential apartment building.",
            "Kitchen fire reported at a restaurant on main road, smoke billowing.",
            "Electric transformer caught fire and exploded, sparks flying.",
            "Gas cylinder leakage and minor fire in shop basement.",
            "Warehouse fire reported in commercial zone, fire engines dispatched."
        ]
    },
    'medical': {
        'priority': 'critical',
        'templates': [
            "Major road accident with multiple injuries, urgent ambulance required.",
            "Citizen collapsed on the footpath showing heart attack symptoms.",
            "Construction site accident, worker fell from third floor with head injury.",
            "Severe burn victim needs immediate medical attention.",
            "Vehicle collision on flyover, passengers trapped and injured."
        ]
    },
    'flooding': {
        'priority': 'high',
        'templates': [
            "Underpass completely flooded with 3 feet of water, two cars stranded.",
            "Heavy rain causing severe waterlogging on main road, traffic blocked.",
            "Stormwater drain choked, rainwater entering residential houses.",
            "Outer ring road flooded, vehicles floating and traffic backed up for miles.",
            "Flooding in basement parking, electrical panels threatened."
        ]
    },
    'water-supply': {
        'priority': 'high',
        'templates': [
            "Main water supply pipeline burst, drinking water overflowing on road.",
            "No water supply in the locality for past 4 consecutive days.",
            "Contaminated, muddy water coming from municipal taps.",
            "BWSSB pipeline leaking high pressure water, flooding the footpath.",
            "Drinking water leakage from main valve near park."
        ]
    },
    'sewage': {
        'priority': 'high',
        'templates': [
            "Sewage overflow from open manhole, spreading bad smell on street.",
            "Blocked sewer line causing backflow into household toilets.",
            "Drainage water overflowing onto public walkway, safety hazard.",
            "Manhole cover broken and sewage bubbling out on main road.",
            "Sewage leakage contaminating nearby fresh water sump."
        ]
    },
    'road-damage': {
        'priority': 'medium',
        'templates': [
            "Large deep pothole on main road causing accidents for two-wheelers.",
            "Road caved in near construction site, needs barricades.",
            "Footpath tiles broken and damaged, unsafe for senior citizens.",
            "Road surface eroded completely after recent heavy rains.",
            "Speed breaker has no paint/markings, causing vehicle damage."
        ]
    },
    'garbage': {
        'priority': 'medium',
        'templates': [
            "Huge pile of garbage dumped near public school entrance, unhygienic.",
            "Garbage bin overflowing, stray dogs and cows spreading waste.",
            "Illegal trash dumping on the corner of the cross road.",
            "Waste accumulated in park, bad odor spreading in neighborhood.",
            "Dry and wet waste mixed and dumped on vacant plot."
        ]
    },
    'traffic': {
        'priority': 'medium',
        'templates': [
            "Traffic signal completely non-functional at busy intersection.",
            "Illegal parking on double lanes causing heavy congestion.",
            "Broken down truck blocking main road lane, heavy jam.",
            "Chaos at crossroad junction due to lack of traffic police.",
            "Road construction work blocking two lanes, severe bottleneck."
        ]
    },
    'streetlight': {
        'priority': 'low',
        'templates': [
            "Streetlight non-functional for past 5 days, dark and unsafe.",
            "Multiple streetlights off on the main avenue, dark spots.",
            "Lamp post bulb flickering constantly, causing visibility issues.",
            "Street lights not switched off during daytime, wasting power.",
            "New streetlight installed but not connected to electricity grid."
        ]
    }
}

wards = ['Mahadevapura', 'Whitefield', 'Koramangala', 'Indiranagar', 'Jayanagar', 
         'Rajajinagar', 'Malleshwaram', 'Basavanagudi', 'Yelahanka', 'Hebbal', 
         'BTM Layout', 'HSR Layout', 'Electronic City', 'Marathahalli', 'JP Nagar']

dataset = []
for _ in range(1200):
    inc_type = random.choice(list(types_and_wards.keys()))
    ward = random.choice(wards)
    info = types_and_wards[inc_type]
    priority = info['priority']
    
    template = random.choice(info['templates'])
    variations = [
        lambda t: t,
        lambda t: t.lower(),
        lambda t: f"Urgent: {t}",
        lambda t: f"Please fix. {t}",
        lambda t: f"Reported at {ward} ward - {t}",
        lambda t: f"{t} Urgent action needed."
    ]
    description = random.choice(variations)(template)
    
    dataset.append({
        'type': inc_type,
        'description': description,
        'ward': ward,
        'priority': priority
    })

df = pd.DataFrame(dataset)
print(f"Generated {len(df)} samples.")

### 4. Build Preprocessor and Transform Features

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(max_features=400, stop_words='english'), 'description'),
        ('cat_type', OneHotEncoder(handle_unknown='ignore'), ['type']),
        ('cat_ward', OneHotEncoder(handle_unknown='ignore'), ['ward'])
    ]
)

X = preprocessor.fit_transform(df).toarray()

priority_classes = ['critical', 'high', 'medium', 'low']
priority_map = {priority_classes[i]: i for i in range(len(priority_classes))}
y_encoded = df['priority'].map(priority_map).values
y = tf.keras.utils.to_categorical(y_encoded, num_classes=4)

# Split
indices = np.random.permutation(len(X))
split_idx = int(0.8 * len(X))
X_train, X_test = X[indices[:split_idx]], X[indices[split_idx:]]
y_train, y_test = y[indices[:split_idx]], y[indices[split_idx:]]

print("Train features shape:", X_train.shape)
print("Test features shape:", X_test.shape)

### 5. Compile Keras Multi-Layer Perceptron (MLP)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X.shape[1],)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(4, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

### 6. Train the Neural Network

In [ ]:
epochs = 15
batch_size = 32

print("Training...")
history = model.fit(
    X_train, y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_test, y_test),
    verbose=1
)

### 7. Evaluate Performance

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {accuracy*100:.2f}%")
print(f"Test Loss: {loss:.4f}")

### 8. Save and Download Model Artifacts
Run this cell to save the model files and automatically trigger Google Colab file downloads so you can download them directly to your computer.

In [ ]:
from google.colab import files

# Save locally in Colab virtual machine
model.save('priority_model.keras')
with open('preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)

print("Artifacts saved. Preparing downloads...")
files.download('priority_model.keras')
files.download('preprocessor.pkl')